#SMOTE


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Preparar datasets (SIN SMOTE)
train_dataset = encoded_data["train"].remove_columns(["id", "text"])
test_dataset = encoded_data["test"].remove_columns(["id", "text"])
dev_dataset = encoded_data["dev"].remove_columns(["id", "text"])

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")
dev_dataset = dev_dataset.rename_column("label", "labels")

# Establecer formato torch
train_dataset.set_format("torch")
test_dataset.set_format("torch")
dev_dataset.set_format("torch")

# Calcular pesos de clase para el conjunto de entrenamiento
train_labels = [example['labels'].item() for example in train_dataset]

print("Distribución original de clases (train):")
unique, counts = np.unique(train_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Clase {label}: {count} ejemplos")

# Calcular pesos balanceados
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

print(f"\nPesos de clase calculados: {class_weights}")
print(f"  Clase 0 (no relevante): peso = {class_weights[0]:.4f}")
print(f"  Clase 1 (relevante): peso = {class_weights[1]:.4f}")


In [ ]:
from transformers import Trainer
from torch import nn
import torch

# Convertir class_weights a tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# Crear Trainer personalizado con pesos de clase
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Usar CrossEntropyLoss con pesos de clase
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

print("✅ WeightedTrainer creado con class weights")


In [ ]:
# Usar el WeightedTrainer en lugar del Trainer normal
trainer = WeightedTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,  # Usar dev_dataset que ya preparamos
    args=args,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer  # Añadir el tokenizer
)

print("✅ Trainer inicializado con class weights")
print(f"📊 Train samples: {len(train_dataset)}")
print(f"📊 Dev samples: {len(dev_dataset)}")


#back translation

In [ ]:
# ============================================================
# BACK TRANSLATION - DATA AUGMENTATION PARA CLASE MINORITARIA
# ============================================================
# Esta celda aumenta los datos de la clase minoritaria (clase 1)
# usando Back Translation (inglés -> otro idioma -> inglés)

!pip install transformers sentencepiece sacremoses -q

import torch
from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
import numpy as np
from datasets import Dataset

class BackTranslation:
    """Clase para realizar back-translation con MarianMT"""

    def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        print(f"🔧 Usando dispositivo: {device}")

    def load_model(self, model_name):
        """Cargar modelo de traducción"""
        print(f"📥 Cargando modelo: {model_name}")
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name).to(self.device)
        return tokenizer, model

    def translate(self, texts, model, tokenizer, batch_size=32):
        """Traducir una lista de textos"""
        translated_texts = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]

            # Tokenizar
            inputs = tokenizer(batch, return_tensors="pt", padding=True,
                              truncation=True, max_length=512).to(self.device)

            # Generar traducción
            with torch.no_grad():
                translated = model.generate(**inputs, max_length=512)

            # Decodificar
            batch_translated = [tokenizer.decode(t, skip_special_tokens=True)
                               for t in translated]
            translated_texts.extend(batch_translated)

        return translated_texts

    def back_translate(self, texts, source_lang="en", target_lang="es"):
        """
        Realizar back-translation: EN -> OTRO IDIOMA -> EN

        Idiomas disponibles:
        - es: Español
        - fr: Francés
        - de: Alemán
        - it: Italiano
        - pt: Portugués
        """
        print(f"\n🔄 BACK TRANSLATION: {source_lang} -> {target_lang} -> {source_lang}")
        print(f"📊 Textos a traducir: {len(texts)}")

        # Paso 1: Traducir a idioma objetivo (EN -> LANG)
        print(f"\n1️⃣ Traduciendo {source_lang} → {target_lang}...")
        forward_model_name = f'Helsinki-NLP/opus-mt-{source_lang}-{target_lang}'
        forward_tokenizer, forward_model = self.load_model(forward_model_name)
        translated = self.translate(texts, forward_model, forward_tokenizer)
        print(f"✅ Traducción completada")

        # Paso 2: Traducir de vuelta (LANG -> EN)
        print(f"\n2️⃣ Traduciendo {target_lang} → {source_lang}...")
        back_model_name = f'Helsinki-NLP/opus-mt-{target_lang}-{source_lang}'
        back_tokenizer, back_model = self.load_model(back_model_name)
        back_translated = self.translate(translated, back_model, back_tokenizer)
        print(f"✅ Traducción de vuelta completada")

        return back_translated

# Inicializar Back Translation
print("🚀 Inicializando Back Translation...")
bt = BackTranslation()

# Obtener solo los textos de la clase minoritaria (clase 1)
print("\n📊 Filtrando textos de clase minoritaria...")
minority_texts = [text for text, label in zip(datasets_loaded['train']['text'],
                                              datasets_loaded['train']['label'])
                  if label == 1]

print(f"✅ Textos de clase minoritaria encontrados: {len(minority_texts)}")

# Realizar back-translation a ESPAÑOL
print("\n" + "="*90)
print("BACK TRANSLATION A ESPAÑOL")
print("="*90)
augmented_spanish = bt.back_translate(minority_texts, source_lang="en", target_lang="es")

# Realizar back-translation a FRANCÉS
print("\n" + "="*90)
print("BACK TRANSLATION A FRANCÉS")
print("="*90)
augmented_french = bt.back_translate(minority_texts, source_lang="en", target_lang="fr")

print("\n" + "="*90)
print("✅ BACK TRANSLATION COMPLETADO")
print("="*90)
print(f"\n📊 RESUMEN DE DATOS AUGMENTADOS:")
print(f"   Textos originales clase 1: {len(minority_texts)}")
print(f"   Textos generados (Spanish): {len(augmented_spanish)}")
print(f"   Textos generados (French): {len(augmented_french)}")
print(f"   Total nuevos textos: {len(augmented_spanish) + len(augmented_french)}")

# Combinar datos originales + augmentados
all_texts = list(datasets_loaded['train']['text']) + augmented_spanish + augmented_french
all_labels = list(datasets_loaded['train']['label']) + [1] * len(augmented_spanish) + [1] * len(augmented_french)

print(f"\n📊 DISTRIBUCIÓN FINAL DEL DATASET AUGMENTADO:")
unique, counts = np.unique(all_labels, return_counts=True)
for label, count in zip(unique, counts):
    ratio = count / counts[0] if label == 1 else 1.0
    print(f"   Clase {label}: {count:,} ejemplos")

original_size = len(datasets_loaded['train']['text'])
new_size = len(all_texts)
print(f"\n📈 Crecimiento del dataset:")
print(f"   Original: {original_size:,} ejemplos")
print(f"   Nuevo: {new_size:,} ejemplos")
print(f"   Incremento: +{new_size - original_size:,} ejemplos ({((new_size-original_size)/original_size)*100:.1f}%)")

# Crear dataset aumentado
print(f"\n🔧 Creando dataset aumentado...")
augmented_dataset = Dataset.from_dict({
    'text': all_texts,
    'label': all_labels
})

print(f"✅ Dataset creado: {len(augmented_dataset)} ejemplos")

# MOSTRAR ALGUNOS EJEMPLOS
print(f"\n" + "="*90)
print("📝 EJEMPLOS DE TEXTOS AUMENTADOS")
print("="*90)

for i in range(min(3, len(minority_texts))):
    print(f"\n🔹 Ejemplo {i+1}:")
    print(f"   Original: {minority_texts[i][:150]}")
    print(f"   ES augmentado: {augmented_spanish[i][:150]}")
    print(f"   FR augmentado: {augmented_french[i][:150]}")


In [ ]:
# ============================================================
# TOKENIZAR DATASET AUMENTADO
# ============================================================

print("🔧 Tokenizando dataset aumentado...")

# Tokenizar el dataset con Back Translation
train_dataset_augmented = augmented_dataset.map(tokenize, batched=True)

# Renombrar columna 'label' a 'labels' (BERT lo necesita así)
train_dataset_augmented = train_dataset_augmented.rename_column("label", "labels")

# Eliminar la columna de texto (ya no la necesitamos)
train_dataset_augmented = train_dataset_augmented.remove_columns(["text"])

# Establecer formato PyTorch
train_dataset_augmented.set_format("torch")

print(f"✅ Train dataset tokenizado: {len(train_dataset_augmented)} ejemplos")

# Preparar dev y test (SIN augmentation - solo los originales)
dev_dataset = encoded_data["dev"].remove_columns(["id", "text"])
test_dataset = encoded_data["test"].remove_columns(["id", "text"])

dev_dataset = dev_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

dev_dataset.set_format("torch")
test_dataset.set_format("torch")

print(f"✅ Dev dataset preparado: {len(dev_dataset)} ejemplos")
print(f"✅ Test dataset preparado: {len(test_dataset)} ejemplos")

# Verificar distribución de clases en el dataset aumentado
import numpy as np
train_labels_augmented = [example['labels'].item() for example in train_dataset_augmented]
unique, counts = np.unique(train_labels_augmented, return_counts=True)

print(f"\n📊 DISTRIBUCIÓN FINAL PARA ENTRENAR:")
for label, count in zip(unique, counts):
    print(f"   Clase {label}: {count:,} ejemplos")

ratio = counts[0] / counts[1]
print(f"   Ratio clase 0/clase 1: {ratio:.2f}:1")

print("\n✅ ¡Todo listo para entrenar con datos aumentados!")


In [ ]:
# ============================================================
# CREAR TRAINER CON DATASET AUMENTADO
# ============================================================

from transformers import Trainer
from transformers import TrainingArguments

# Preparar dev_dataset si no lo tienes ya
if 'dev_dataset' not in locals():
    dev_dataset = encoded_data["dev"].remove_columns(["id", "text"])
    dev_dataset = dev_dataset.rename_column("label", "labels")
    dev_dataset.set_format("torch")

# Actualizar TrainingArguments para deshabilitar wandb correctamente
args = TrainingArguments(
    output_dir="./outputs",
    report_to="none",  # 🔧 AQUÍ: Deshabilita wandb correctamente
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch"
)

# Crear Trainer con datos aumentados
trainer = Trainer(
    model=model,
    train_dataset=train_dataset_augmented,  # ⭐ DATASET AUMENTADO
    eval_dataset=dev_dataset,
    args=args,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

print("✅ Trainer creado con dataset aumentado")
print(f"📊 Entrenará con {len(train_dataset_augmented):,} ejemplos")
